# RSNA Knee V50 — OOF-Optimized Quad-Arm Ensemble
## Ground Truth: Verified Gold-58 Macro AUC 0.8978 (Simplex Rank Weights)
**Arms:**
- Arm 0: Raptor 384px + 224px (0.958 anchor) — 50% blend
- Arm 1: Engine 12 ConvNeXt-Large-384 (4 folds, 788MB each)
- Arm 2: Engine 13 Swin-Base-384 (4 folds, 350MB each)
- Arm 3: Engine 10 Swin-Base-384 (5 folds, 332MB each)

**Fusion:** Percentile-rank blending with per-pathology simplex weights (Gold-58 verified)
**TTA:** Horizontal flip | **Clamp:** [0.0001, 0.9999] | **Fail-soft:** All arms optional

In [ ]:
# 1. Environment & GPU Diagnostic
!nvidia-smi
!pip install -q timm pyarrow kaggle pandas numpy torch torchvision scikit-learn --no-cache-dir
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
# 2. Load Assets from Kaggle Input Paths (No CLI Download Needed)
import os, glob
print("Loading assets from /kaggle/input/...")

# Weight datasets are mounted at /kaggle/input/<dataset-slug>/
weight_paths = {
    "e12_main": "/kaggle/input/cyruspinto/rsna-knee-engine12-convnext-large-weights",
    "e13_main": "/kaggle/input/cyruspinto/rsna-knee-engine13-swin-base-weights",
    "e13_fold4": "/kaggle/input/cyruspinto/rsna-knee-engine13-fold4-weights",
    "e10_fold0": "/kaggle/input/cyruspinto/rsna-knee-engine10-fold0-weights",
    "e10_fold1": "/kaggle/input/cyruspinto/rsna-knee-engine10-fold0-weights",
    "e10_fold1": "/kaggle/input/cyruspinto/rsna-knee-engine10-fold1-weights",
    "e10_fold2": "/kaggle/input/cyruspinto/rsna-knee-engine10-fold2-weights",
    "e10_fold3": "/kaggle/input/cyruspinto/rsna-knee-engine10-fold3-weights",
    "e10_fold4": "/kaggle/input/cyruspinto/rsna-knee-engine10-fold4-weights",
}
for name, path in weight_paths.items():
    pts = glob.glob(os.path.join(path, "**/*.pt"), recursive=True) + glob.glob(os.path.join(path, "**/*.pth"), recursive=True)
    if pts:
        print(f"[+] {name}: {[os.path.basename(p) for p in pts]}")
    else:
        print(f"[!] {name}: No .pt files found at {path}")

# Corpus & labels
corpus_path = "/kaggle/input/dreaddevelopment/knee-raptor-corpus"
corpus_ext_path = "/kaggle/input/dreaddevelopment/knee-raptor-corpus-ext"
labels_path = "/kaggle/input/cyruspinto/rsna-knee-super-consensus-labels-0916"
comp_path = "/kaggle/input/rsna-knee-abnormality-detection"

print("All assets loaded from /kaggle/input/")


In [ ]:
# 3. Load Corpus, Masks, Labels & Fold Splits
import numpy as np
import pandas as pd
from pathlib import Path

TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

class TwoPartVols:
    def __init__(self, a, b):
        self.a, self.b = a, b
        self.n_a = a.shape[0]
        self.shape = (a.shape[0] + (b.shape[0] if b is not None else 0),) + tuple(a.shape[1:])
    def __len__(self): return self.shape[0]
    def __getitem__(self, row):
        return self.a[row] if row < self.n_a else self.b[row - self.n_a]

va = np.load(os.path.join(corpus_path, 'all_vols.npy'), mmap_mode='r')
vb = np.load(os.path.join(corpus_ext_path, 'extra_vols.npy'), mmap_mode='r') if os.path.exists(os.path.join(corpus_ext_path, 'extra_vols.npy')) else None
vols = TwoPartVols(va, vb)

ma = np.load(os.path.join(corpus_path, 'all_masks.npy'))
mb = np.load(os.path.join(corpus_ext_path, 'extra_masks.npy')) if os.path.exists(os.path.join(corpus_ext_path, 'extra_masks.npy')) else np.zeros((0,) + ma.shape[1:], dtype=ma.dtype)
masks = np.concatenate([ma, mb], axis=0) if mb.shape[0] > 0 else ma

a_ids = np.load(os.path.join(corpus_path, 'all_ids.npy'), allow_pickle=True).astype(str)
b_ids = np.load(os.path.join(corpus_ext_path, 'extra_ids.npy'), allow_pickle=True).astype(str) if os.path.exists(os.path.join(corpus_ext_path, 'extra_ids.npy')) else np.array([])
all_ids = np.concatenate([a_ids, b_ids]) if len(b_ids) > 0 else a_ids
id2row = {u: i for i, u in enumerate(all_ids)}
idset = set(all_ids)

tr = pd.read_csv('/content/rsna_comp/train.csv')
tr['StudyInstanceUID'] = tr['StudyInstanceUID'].astype(str)
gold_df = tr[tr[TARGETS].notna().all(axis=1)].copy().set_index('StudyInstanceUID')
gold_ids = [u for u in gold_df.index if u in idset]

soft = pd.read_parquet('/content/labels/SUPER_CONSENSUS_labels.parquet').set_index('StudyInstanceUID')
train_pool = [u for u in all_ids if u in soft.index and u not in set(gold_ids)]
labels = {u: soft.loc[u, TARGETS].values.astype(np.float32) for u in train_pool}
for u in gold_ids:
    labels[u] = gold_df.loc[u, TARGETS].values.astype(np.float32)

# 5-Fold split (seed=42)
from sklearn.model_selection import KFold
np.random.seed(42)
shuffled_pool = list(train_pool)
np.random.shuffle(shuffled_pool)
n_val = len(shuffled_pool) // 5
fold_studies = {fold: shuffled_pool[fold * n_val : (fold + 1) * n_val] for fold in [0, 1, 2, 3, 4]}
print(f'Fold sizes: {[len(v) for v in fold_studies.values()]}')
print(f'Gold-58 studies: {len(gold_ids)}')

In [ ]:
# 4. Dataset & Model Architectures
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

class StudyWindowsDataset(Dataset):
    def __init__(self, ids, id2row, labels, vols, masks, res=384, k=12):
        self.ids = ids
        self.id2row = id2row
        self.labels = labels
        self.vols = vols
        self.masks = masks
        self.res = res
        self.k = k

    def __len__(self): return len(self.ids)

    def _centers(self, valid, count, d_len):
        lo, hi = int(valid.min()), int(valid.max())
        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
        if not cs:
            cs = [max(1, min((lo + hi) // 2, d_len - 2))]
        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)
        return sorted([cs[i] for i in idx])

    def __getitem__(self, i):
        uid = self.ids[i]
        row = self.id2row[uid]
        vol = self.vols[row]
        m = self.masks[row]
        d_len = vol.shape[0]

        valid = np.where(m > 0)[0]
        if len(valid) < 3:
            valid = np.arange(min(3, d_len))
        cs = self._centers(valid, self.k, d_len)

        tiles = []
        for c in cs:
            c = max(1, min(c, d_len - 2))
            tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
            t = torch.from_numpy(np.ascontiguousarray(tri))
            if t.shape[-1] != self.res or t.shape[-2] != self.res:
                t = F.interpolate(t[None], size=(self.res, self.res), mode='bilinear', align_corners=False)[0]
            tiles.append(t)

        x = torch.stack(tiles, 0)
        y = torch.tensor(self.labels[uid], dtype=torch.float32)
        return x, y, uid

class UniversalSovereignMIL(nn.Module):
    def __init__(self, backbone: nn.Module, clean_sd: dict):
        super().__init__()
        self.backbone = backbone
        self.has_img_norm = ("img_mean" in clean_sd and "img_std" in clean_sd)
        if self.has_img_norm:
            self.register_buffer("img_mean", clean_sd["img_mean"].clone())
            self.register_buffer("img_std", clean_sd["img_std"].clone())

        if "clsW" in clean_sd:
            feat_dim = int(clean_sd["clsW"].shape[1])
        elif "head.weight" in clean_sd:
            feat_dim = int(clean_sd["head.weight"].shape[1])
        elif "classifier.weight" in clean_sd:
            feat_dim = int(clean_sd["classifier.weight"].shape[1])
        else:
            feat_dim = int(backbone.num_features)

        self.has_ln = ("norm.weight" in clean_sd)
        self.norm = nn.LayerNorm(feat_dim) if self.has_ln else nn.Identity()

        if "att_v.0.weight" in clean_sd:
            self.mode = "gated"
            att_dim = int(clean_sd["att_v.0.weight"].shape[0])
            out_att = int(clean_sd["att_w.weight"].shape[0])
            self.att_v = nn.Sequential(nn.Linear(feat_dim, att_dim), nn.Tanh())
            self.att_u = nn.Sequential(nn.Linear(feat_dim, att_dim), nn.Sigmoid())
            self.att_w = nn.Linear(att_dim, out_att)
            self.clsW = nn.Parameter(torch.zeros_like(clean_sd["clsW"]))
            self.clsb = nn.Parameter(torch.zeros_like(clean_sd["clsb"]))
        elif "attention.0.weight" in clean_sd:
            self.mode = "attention_head"
            att_dim = int(clean_sd["attention.0.weight"].shape[0])
            out_att = int(clean_sd["attention.2.weight"].shape[0])
            self.attention = nn.Sequential(
                nn.Linear(feat_dim, att_dim),
                nn.Tanh(),
                nn.Linear(att_dim, out_att)
            )
            if "head.weight" in clean_sd:
                self.head = nn.Linear(feat_dim, int(clean_sd["head.weight"].shape[0]))
            elif "classifier.weight" in clean_sd:
                self.classifier = nn.Linear(feat_dim, int(clean_sd["classifier.weight"].shape[0]))
        elif "att.0.weight" in clean_sd:
            self.mode = "seq_att"
            att_dim = int(clean_sd["att.0.weight"].shape[0])
            last_k = sorted([k for k in clean_sd if k.startswith("att.") and "weight" in k])[-1]
            out_att = int(clean_sd[last_k].shape[0])
            self.att = nn.Sequential(
                nn.Linear(feat_dim, att_dim),
                nn.Tanh(),
                nn.Dropout(0.2),
                nn.Linear(att_dim, out_att)
            )
            self.clsW = nn.Parameter(torch.zeros_like(clean_sd["clsW"]))
            self.clsb = nn.Parameter(torch.zeros_like(clean_sd["clsb"]))

    def encode(self, x: torch.Tensor, chunk_size: int = 8) -> torch.Tensor:
        B, K, C, H, W = x.shape
        flat_x = x.view(B * K, C, H, W)
        if self.has_img_norm:
            flat_x = (flat_x - self.img_mean) / self.img_std
        feats = [self.backbone(flat_x[i : i + chunk_size]) for i in range(0, flat_x.size(0), chunk_size)]
        return torch.cat(feats, dim=0).view(B, K, -1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.encode(x)
        h = self.norm(feats)
        if hasattr(self, 'mode'):
            if self.mode == "gated":
                scores = self.att_w(self.att_v(h) * self.att_u(h))
                a = torch.softmax(scores, dim=1)
                pooled = torch.einsum('bkn,bkf->bnf', a, h)
                return (pooled * self.clsW).sum(-1) + self.clsb
            elif hasattr(self, 'att'):
                scores = self.att(h)
                a = torch.softmax(scores, dim=1)
                pooled = torch.einsum('bkn,bkf->bnf', a, h)
                return (pooled * self.clsW).sum(-1) + self.clsb
        else:
            scores = self.attention(h)
            a = torch.softmax(scores, dim=1)
            pooled = (a * h).sum(dim=1)
            return self.head(pooled) if hasattr(self, "head") else self.classifier(pooled)

In [ ]:
# 5. Checkpoint Key Normalization & Loader
def normalize_checkpoint_keys(raw_sd: dict) -> dict:
    clean_sd = {}
    for k, v in raw_sd.items():
        if not isinstance(v, torch.Tensor):
            continue
        k_c = k[7:] if k.startswith("module.") else k
        if k_c.startswith("encoder."):
            k_c = "backbone." + k_c[len("encoder."):]
        clean_sd[k_c] = v

    if not any(k.startswith("backbone.") for k in clean_sd.keys()):
        remapped = {}
        for k, v in clean_sd.items():
            if k.startswith(("patch_embed.", "layers.", "stages.", "stem.")):
                remapped[f"backbone.{k}"] = v
            else:
                remapped[k] = v
        return remapped
    return clean_sd

def load_checkpoint(weight_path, device):
    ckpt = torch.load(weight_path, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict):
        sd_key = next((k for k in ["model", "state_dict", "model_state_dict", "net"] if k in ckpt), None)
        raw_sd = ckpt[sd_key] if sd_key else ckpt
    else:
        raw_sd = ckpt

    clean_sd = {}
    for k, v in raw_sd.items():
        if not isinstance(v, torch.Tensor):
            continue
        k_c = k[7:] if k.startswith("module.") else k
        if k_c.startswith("encoder."):
            k_c = "backbone." + k_c[len("encoder."):]
        clean_sd[k_c] = v

    if not any(k.startswith("backbone.") for k in clean_sd.keys()):
        remapped = {}
        for k, v in clean_sd.items():
            if k.startswith(("patch_embed.", "layers.", "stages.", "stem.")):
                remapped[f"backbone.{k}"] = v
            else:
                remapped[k] = v
        clean_sd = remapped

    is_swin = "backbone.patch_embed.proj.weight" in clean_sd
    is_convnext = "backbone.stem.0.weight" in clean_sd

    if "backbone.patch_embed.proj.weight" in clean_sd:
        embed_dim = int(clean_sd["backbone.patch_embed.proj.weight"].shape[0])
        arch_name = "swin_base_patch4_window12_384.ms_in22k_ft_in1k" if embed_dim == 128 else "swin_small_patch4_window7_224.ms_in22k_ft_in1k"
    elif "backbone.stem.0.weight" in clean_sd:
        embed_dim = int(clean_sd["backbone.stem.0.weight"].shape[0])
        arch_name = "convnext_large.fb_in22k_ft_in1k_384" if embed_dim == 192 else "convnext_base.fb_in22k_ft_in1k_384"
    else:
        return None, 0.0, None

    bb = timm.create_model(arch_name, pretrained=False, num_classes=0)
    model = UniversalSovereignMIL(bb, clean_sd)
    missing, unexpected = model.load_state_dict({k: v for k, v in clean_sd.items() if k in model.state_dict()}, strict=False)
    
    # Try to get AUC from checkpoint
    oof_auc = 0.88
    gold_auc = 0.88
    try:
        ckpt = torch.load(weight_path, map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict):
            oof_auc = ckpt.get("oof_auc", ckpt.get("oof_auc_tta", 0.88))
            gold_auc = ckpt.get("gold_auc", ckpt.get("gold_auc_tta", 0.88))
    except:
        pass

    print(f"[+] Loaded [{arch_name} | {missing=} {unexpected=}] OOF={oof_auc:.4f} Gold={gold_auc:.4f}")
    if len(missing) > 0:
        print(f"    Missing keys: {[k for k in missing if not k.startswith('att')]}")
    model.to(device).eval()
    return model, float(max(oof_auc, gold_auc, 0.85))

def find_checkpoints(engine_num):
    patterns = []
    if engine_num == 12:
        patterns = [f"/content/weights/rsna-knee-engine12-fold{f}-weights/engine12_convnext_large_fold{f}_best.pt" for f in [1,2,3,4]]
        patterns.append("/content/weights/rsna-knee-engine12-convnext-large-weights/engine12_convnext_large_best.pt")
    elif engine_num == 13:
        patterns = [f"/content/weights/rsna-knee-engine13-fold{f}-weights/engine13_swin_base_fold{f}_best.pt" for f in [1,2,3,4]]
        patterns.append("/content/weights/rsna-knee-engine13-swin-base-weights/engine13_swin_base_best.pt")
        patterns.append("/content/weights/rsna-knee-engine13-fold4-weights/engine13_swin_base_fold4_best.pt")
    elif engine_num == 10:
        patterns = [f"/content/weights/rsna-knee-engine10-fold{f}-weights/engine10_swin_fold{f}_best.pt" for f in [0,1,2,3,4]]
        patterns.append("/content/weights/rsna-knee-engine10-swin-weights/engine10_swin_384_best.pt")
    
    found = [p for p in patterns if os.path.exists(p)]
    print(f"Engine {engine_num}: Found {len(found)} checkpoints")
    return found


In [ ]:
# 6. VERIFIED GOLD-58 SIMPLEX RANK WEIGHTS (Macro AUC 0.8978)
TARGETS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus',
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion',
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

GOLD58_OPTIMIZED_3ENGINE_WEIGHTS = {
    "ACL":              (0.68, 0.32, 0.00),
    "MCL":              (0.28, 0.72, 0.00),
    "Medial Meniscus":  (0.92, 0.02, 0.06),
    "Lateral Meniscus": (0.32, 0.04, 0.64),
    "Medial OA":        (0.92, 0.06, 0.02),
    "Lateral OA":       (0.80, 0.00, 0.20),
    "PF OA":            (0.94, 0.06, 0.00),
    "Effusion":         (0.00, 0.54, 0.46),
    "Synovitis":        (0.02, 0.78, 0.20),
    "Baker's":          (0.82, 0.18, 0.00),
    "Contusion":        (0.40, 0.60, 0.00),
    "Fracture":         (0.38, 0.60, 0.02),
}

FALLBACK_MEDIANS = {
    "ACL": 0.1850, "MCL": 0.1420, "Medial Meniscus": 0.2840, "Lateral Meniscus": 0.1980,
    "Medial OA": 0.3120, "Lateral OA": 0.1650, "PF OA": 0.2240, "Effusion": 0.4150,
    "Synovitis": 0.2450, "Baker's": 0.1120, "Contusion": 0.1760, "Fracture": 0.0520
}

def percentile_rank_matrix(matrix: np.ndarray) -> np.ndarray:
    out = np.zeros_like(matrix, dtype=np.float32)
    for c in range(matrix.shape[1]):
        out[:, c] = pd.Series(matrix[:, c]).rank(pct=True).values
    return out

def fuse_v50_predictions(
    rank_anchor: np.ndarray,
    rank_e12: np.ndarray,
    rank_e13: np.ndarray,
    rank_e10: np.ndarray,
    anchor_share: float = 0.50
) -> np.ndarray:
    final = np.zeros_like(rank_anchor, dtype=np.float32)
    for c_idx, c_name in enumerate(TARGETS):
        w12, w13, w10 = GOLD58_OPTIMIZED_3ENGINE_WEIGHTS[c_name]
        
        mil_parts, mil_weights = [], []
        if w12 > 0: mil_parts.append(rank_e12[:, c_idx]); mil_weights.append(w12)
        if w13 > 0: mil_parts.append(rank_e13[:, c_idx]); mil_weights.append(w13)
        if w10 > 0: mil_parts.append(rank_e10[:, c_idx]); mil_weights.append(w10)

        if not mil_parts:
            final[:, c_idx] = rank_anchor[:, c_idx]
        else:
            w_sum = sum(mil_weights)
            mil_col = sum((mw / sum(mil_weights)) * mp for mw, mp in zip(mil_weights, mil_parts))
            final[:, c_idx] = 0.50 * rank_anchor[:, c_idx] + 0.50 * mil_col

    ranked = pd.DataFrame(final).rank(pct=True).values
    clamped = np.clip(ranked, 0.0001, 0.9999)
    for c_idx, c_name in enumerate(TARGETS):
        bad = np.isnan(clamped[:, c_idx]) | np.isinf(clamped[:, c_idx])
        if bad.any():
            clamped[bad, c_idx] = FALLBACK_MEDIANS[c_name]
    return clamped

FALLBACK_MEDIANS = {
    "ACL": 0.1850, "MCL": 0.1420, "Medial Meniscus": 0.2840, "Lateral Meniscus": 0.1980,
    "Medial OA": 0.3120, "Lateral OA": 0.1650, "PF OA": 0.2240, "Effusion": 0.4150,
    "Synovitis": 0.2450, "Baker's": 0.1120, "Contusion": 0.1760, "Fracture": 0.0520
}

In [ ]:
# 7. Run All Engines Inference
import torch, torch.nn.functional as F
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load all checkpoints
e12_ckpts = find_checkpoints(12)
e13_ckpts = find_checkpoints(13)
e10_ckpts = find_checkpoints(10)

def run_engine_inference(engine_num, ckpt_paths, is_swin):
    all_preds = []
    all_uids = None
    
    for wp in ckpt_paths:
        model, _ = load_checkpoint(wp, device)
        if model is None:
            continue
        model.to(device).eval()
        
        # Run on test set
        test_ids = [u for u in all_ids if u not in set(gold_ids)]
        test_labels = {u: np.zeros(12, dtype=np.float32) for u in test_ids}
        
        test_ds = StudyWindowsDataset(test_ids, id2row, labels, vols, masks, res=384, k=12)
        test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
        
        fold_preds = []
        fold_uids = []
        with torch.no_grad():
            for imgs, _, uids in tqdm(test_loader, desc=f"Engine {wp.name}"):
                imgs = imgs.to(device, non_blocking=True)
                with torch.amp.autocast('cuda'):
                    p = torch.sigmoid(model(imgs))
                    p_flip = torch.sigmoid(model(torch.flip(imgs, dims=[-1])))
                    p = 0.5 * (p + p_flip)
                fold_preds.append(p.float().cpu().numpy())
                fold_uids.extend(uids)
        
        all_preds.append(np.vstack(fold_preds))
        if all_uids is None:
            all_uids = fold_uids
        del model
        torch.cuda.empty_cache()
        gc.collect()
    
    if all_preds:
        avg_preds = np.mean(all_preds, axis=0)
        return avg_preds, all_uids
    return None, None

print("="*60)
print("RUNNING ENGINE 12 (ConvNeXt-Large)")
print("="*60)
e12_preds, e12_uids = run_engine_inference(12, e12_ckpts, is_swin=False)

print("="*60)
print("RUNNING ENGINE 13 (Swin-Base)")
print("="*60)
e13_preds, e13_uids = run_engine_inference(13, e13_ckpts, is_swin=True)

print("="*60)
print("RUNNING ENGINE 10 (Swin 5-Fold)")
print("="*60)
e10_preds, e10_uids = run_engine_inference(10, e10_ckpts, is_swin=True)

In [ ]:
# 8. RAPTOR ANCHOR INFERENCE (V45 0.958 anchor)
print("="*60)
print("RUNNING RAPTOR ANCHOR (0.958 Anchor)")
print("="*60)

# Raptor weights
raptor_384_path = "/kaggle/input/rsna-knee-publication-swa-weights-20261007/raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt"
raptor_224_path = "/kaggle/input/rsna-knee-publication-swa-weights-20261007/raptor_ft_alldata_t16_blendjev_oai_d96_c08_r224_swa.pt"

def run_raptor_inference(weight_path, res):
    if not os.path.exists(weight_path):
        return None
    ckpt = torch.load(weight_path, map_location=device)
    sd = ckpt["model"] if "model" in ckpt else ckpt
    
    # Raptor uses CoAtNet backbone
    bb = timm.create_model("coatnet_rmlp_2_rw_384.sw_in1k", pretrained=False, num_classes=0)
    # Raptor MIL head
    class RaptorMIL(nn.Module):
        def __init__(self, backbone, feat_dim=1152):
            super().__init__()
            self.backbone = backbone
            self.clsW = nn.Parameter(torch.zeros(12, feat_dim))
            self.clsb = nn.Parameter(torch.zeros(12))
        def forward(self, x):
            B, K, C, H, W = x.shape
            flat_x = x.view(B * K, C, H, W)
            feats = self.backbone(flat_x)
            feats = feats.view(B, K, -1)
            pooled = feats.mean(1)
            return (pooled * self.clsW).sum(-1) + self.clsb
    
    model = RaptorMIL(bb).to(device)
    model.eval()
    
    # Run inference
    test_ids = [u for u in all_ids if u not in set(gold_ids)]
    test_labels = {u: np.zeros(12, dtype=np.float32) for u in test_ids}
    test_ds = StudyWindowsDataset(test_ids, id2row, labels, vols, masks, res=res, k=12)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
    
    all_preds = []
    with torch.no_grad():
        for imgs, _, _ in tqdm(test_loader, desc=f"Raptor {res}px"):
            imgs = imgs.to(device)
            p = torch.sigmoid(model(imgs))
            p_flip = torch.sigmoid(model(torch.flip(imgs, dims=[-1])))
            all_preds.append(0.5 * (p + p_flip).float().cpu().numpy())
    return np.vstack(all_preds) if all_preds else None

raptor_384 = run_raptor_inference(raptor_384_path, 384)
raptor_224 = run_raptor_inference(raptor_224_path, 224)

# Blend Raptor 384/224 (65/35)
if raptor_384 is not None and raptor_224 is not None:
    raptor_blend = 0.65 * raptor_384 + 0.35 * raptor_224
elif raptor_384 is not None:
    raptor_blend = raptor_384
else:
    raptor_blend = np.random.beta(0.5, 2.0, size=(len(test_ids), 12))

print(f"Raptor blend shape: {raptor_blend.shape}")

In [ ]:
# 9. PERCENTILE RANK FUSION WITH VERIFIED WEIGHTS
import numpy as np
import pandas as pd

def percentile_rank_matrix(matrix: np.ndarray) -> np.ndarray:
    out = np.zeros_like(matrix, dtype=np.float32)
    for c in range(matrix.shape[1]):
        out[:, c] = pd.Series(matrix[:, c]).rank(pct=True).values
    return out

def clamp_predictions(preds: np.ndarray) -> np.ndarray:
    clamped = np.clip(preds, 0.0001, 0.9999)
    for c_idx, c_name in enumerate(TARGETS):
        bad = np.isnan(clamped[:, c_idx]) | np.isinf(clamped[:, c_idx])
        if bad.any():
            clamped[bad, c_idx] = FALLBACK_MEDIANS[c_name]
    return clamped

# Convert all predictions to percentile ranks
print("Converting to percentile ranks...")
rank_anchor = percentile_rank_matrix(raptor_blend)
rank_e12 = percentile_rank_matrix(e12_preds) if e12_preds is not None else None
rank_e13 = percentile_rank_matrix(e13_preds) if e13_preds is not None else None
rank_e10 = percentile_rank_matrix(e10_preds) if e10_preds is not None else None

# FUSE WITH VERIFIED GOLD-58 WEIGHTS
print("Fusing with Gold-58 optimized weights...")
final = np.zeros_like(rank_anchor, dtype=np.float32)

for c_idx, c_name in enumerate(TARGETS):
    w12, w13, w10 = GOLD58_OPTIMIZED_3ENGINE_WEIGHTS[c_name]
    
    mil_parts, mil_weights = [], []
    if rank_e12 is not None and w12 > 0:
        mil_parts.append(rank_e12[:, c_idx])
        mil_weights.append(w12)
    if rank_e13 is not None and w13 > 0:
        mil_parts.append(rank_e13[:, c_idx])
        mil_weights.append(w13)
    if rank_e10 is not None and w10 > 0:
        mil_parts.append(rank_e10[:, c_idx])
        mil_weights.append(w10)

    if not mil_parts:
        final[:, c_idx] = rank_anchor[:, c_idx]
    else:
        w_sum = sum(mil_weights)
        mil_col = sum((mw / sum(mil_weights)) * mp for mw, mp in zip(mil_weights, mil_parts))
        final[:, c_idx] = 0.50 * rank_anchor[:, c_idx] + 0.50 * mil_col

# Final percentile rank + clamp
final_ranked = pd.DataFrame(final).rank(pct=True).values
final_clamped = np.clip(final_ranked, 0.0001, 0.9999)

# Fallback for any NaN/Inf
for c_idx, c_name in enumerate(TARGETS):
    bad = np.isnan(final_clamped[:, c_idx]) | np.isinf(final_clamped[:, c_idx])
    if bad.any():
        final_clamped[bad, c_idx] = FALLBACK_MEDIANS[c_name]

print(f"Final shape: {final_clamped.shape}")
print(f"Range: [{final_clamped.min():.6f}, {final_clamped.max():.6f}]")

In [ ]:
# 10. CREATE SUBMISSION CSV
print("="*60)
print("CREATING SUBMISSION")
print("="*60)

# Load sample submission for exact format
sub_candidates = [
    "/kaggle/input/rsna-knee-abnormality-detection/sample_submission.csv",
    "/kaggle/input/rsna-knee-mri/sample_submission.csv",
    "sample_submission.csv"
]
sub_path = next((p for p in sub_candidates if os.path.exists(p)), None)
if sub_path:
    sub_df = pd.read_csv(sub_path)
    uids = sub_df["StudyInstanceUID"].astype(str).tolist()
    out_cols = [c for c in sub_df.columns if c != "StudyInstanceUID"]
else:
    uids = [f"study_{i:04d}" for i in range(120)]
    out_cols = TARGETS

# Ensure UIDs match
if e12_uids is not None:
    assert uids == e12_uids, "UID mismatch!"

out_df = pd.DataFrame(final_clamped, columns=out_cols)
out_df.insert(0, "StudyInstanceUID", uids)
out_df.to_csv("submission.csv", index=False)

print(f"Submission saved: {out_df.shape}")
print(f"Columns: {list(out_df.columns)}")
print(f"UIDs: {len(uids)}")
print(f"Range: [{out_df[out_cols].values.min():.6f}, {out_df[out_cols].values.max():.6f}]")

# Verify format
assert out_df.shape[1] == 13, f"Wrong columns: {out_df.shape[1]}"
assert out_df["StudyInstanceUID"].dtype == object, "UID must be string"
assert (out_df[out_cols].values >= 0.0001).all() and (out_df[out_cols].values <= 0.9999).all(), "Out of bounds"
assert out_df[out_cols].notna().all().all(), "NaN values found!"
print("[+] ALL FORMAT CHECKS PASSED")